In [ ]:
run_id = "00000000-0000-0000-0000-000000000000"
bucket = "django-prefect-datalake-dev"
aws_access_key_id = "rustfs"
aws_secret_access_key = "rustfs_secret"
aws_s3_region = "us-east-1"
s3_endpoint = "localhost:9000"
notebook_output_dir = "data/notebook_outputs"

In [ ]:
import json
import s3fs
import polars as pl

In [ ]:
endpoint_url = f"http://{s3_endpoint}" if not s3_endpoint.startswith("http") else s3_endpoint
storage_options = {
    "key": aws_access_key_id,
    "secret": aws_secret_access_key,
    "endpoint_url": endpoint_url,
}
s3 = s3fs.S3FileSystem(key=aws_access_key_id, secret=aws_secret_access_key,
                       endpoint_url=endpoint_url,
                       client_kwargs={"region_name": aws_s3_region})

In [ ]:
input_path = f"s3://{bucket}/processed/flows/data-processing/{run_id}/02_validated.parquet"
df = pl.scan_parquet(input_path, storage_options=storage_options)

In [ ]:
df = (
    df
    .with_columns([
        pl.col('transaction_date').str.strptime(pl.Date, '%Y-%m-%d').alias('date'),
        (pl.col('amount') * pl.col('quantity')).alias('total'),
        (pl.col('amount') * 0.1).alias('tax'),
    ])
    .with_columns([
        pl.col('date').dt.strftime('%Y-%m').alias('year_month'),
        pl.when(pl.col('total') > 1000)
          .then(pl.lit('high'))
          .when(pl.col('total') > 100)
          .then(pl.lit('medium'))
          .otherwise(pl.lit('low'))
          .alias('amount_category'),
    ])
)

In [ ]:
output_path = f"s3://{bucket}/processed/flows/data-processing/{run_id}/03_transformed.parquet"

with s3.open(output_path, "wb") as f:
    df.collect().write_parquet(f, compression="snappy", statistics=True, use_pyarrow=True)

row_count = df.select(pl.len()).collect().item()
print(f"Transformed {row_count} rows → {output_path}")
print(json.dumps({"step": "transform", "row_count": row_count, "output": output_path}))